# EvolveAI — T4 Max-GPU One-Shot SNN Training

One full training invocation. Four approved Hugging Face streams feed the bounded Gatekeeper; mixed precision and a larger recurrent SNN configuration increase useful GPU work. This remains a research prototype, not AGI or a conscious system.

**This notebook does not run benchmark sweeps or repeated test jobs.** Validation evaluations are retained inside the single training run for early stopping and checkpoint selection. Colab GPU quotas and session limits still apply.

In [ ]:
from pathlib import Path
import subprocess, sys
REPO_DIR = Path('/content/EvolveAI')
BRANCH = 'evolveai-maxgpu-one-shot'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/swasthik-s/EvolveAI.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
%cd /content/EvolveAI/snn
%pip -q install 'datasets>=3.0,<5.0'
print('Repository branch:', BRANCH)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Select Runtime > Change runtime type > T4 GPU.')
props = torch.cuda.get_device_properties(0)
print('PyTorch:', torch.__version__)
print('GPU:', props.name)
print(f'VRAM: {props.total_memory / 1024**3:.2f} GiB')
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('CUDA:', torch.version.cuda)
print('This run uses automatic BF16/FP16 mixed precision; T4 normally selects FP16.')


## Persistent storage

Mount Drive before training. Checkpoints, report and episodic memory are written there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/EvolveAI/maxgpu_run')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)


## Brain v2 continuation run

This version cycles finite Hugging Face streams, uses a stable content-hash train/validation split, reports CUDA allocated/reserved/peak memory, and reduces director log volume. It warm-starts from the previous best model weights; optimizer state is not restored, so this is fine-tuning rather than exact optimizer resume.

Do not overwrite the original checkpoint. The run writes to a new `brain_v2_run` directory.

In [ ]:
# ONE TRAINING JOB — warm-start from the existing best checkpoint
import importlib
import train, streaming_gatekeeper, streaming_train, streaming_one_shot
importlib.reload(train)
importlib.reload(streaming_gatekeeper)
importlib.reload(streaming_train)
importlib.reload(streaming_one_shot)
from streaming_one_shot import run_streaming_one_shot

SOURCE_IDS = ['tinystories', 'wikitext', 'ag_news', 'gsm8k']
PREVIOUS_BEST = Path('/content/drive/MyDrive/EvolveAI/maxgpu_run/streaming_one_shot_checkpoint.pt')
RUN_DIR = Path('/content/drive/MyDrive/EvolveAI/brain_v2_run')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not PREVIOUS_BEST.is_file():
    raise FileNotFoundError(f'Previous best checkpoint not found: {PREVIOUS_BEST}')
print('Approved streams:', SOURCE_IDS)
print('Warm-starting from:', PREVIOUS_BEST)
print('Output directory:', RUN_DIR)

report = run_streaming_one_shot(
    output_dir=RUN_DIR,
    source_ids=SOURCE_IDS,
    steps=8000,
    eval_interval=200,
    patience=15,
    min_delta=0.001,
    block_size=128,
    batch_size=64,
    hidden_size=256,
    embedding_dim=96,
    learning_rate=0.0005,
    seed=43,
    architecture='baseline',
    device='cuda',
    resume_checkpoint_path=PREVIOUS_BEST,
)

print('\n=== BRAIN V2 REPORT ===')
print('Steps:', report['steps_completed'])
print('Best step:', report['best_step'])
print('Best validation loss:', report['best_validation_loss'])
print('Warm started from:', report['warm_started_from'])
print('Report:', RUN_DIR / 'streaming_one_shot_report.json')
print('Best checkpoint:', report['checkpoint_path'])


In [ ]:
import json
report = json.loads((RUN_DIR / 'streaming_one_shot_report.json').read_text(encoding='utf-8'))
print('Configuration:', report['training_config'])
print('Warm-start source:', report.get('warm_started_from'))
print('Gatekeeper:', report['gatekeeper'])
print('Memory events:', report['episodic_memory']['events'])
print('Learning director decisions:', report['learning_director']['decisions'])
print('Final GPU telemetry:', report['training_history'][-1].get('gpu_memory'))


## Interpretation and limitations

- The Gatekeeper streams records; it does not download entire datasets.
- The model predicts the next byte. It is not yet a structured reasoning or conversational model.
- The online validation buffer is useful for monitoring but is not an independent benchmark.
- A checkpoint is a trained model state, not a complete resumable training session with optimizer and stream cursor state.
- GPU utilization depends on Colab's assigned accelerator, streaming/network throughput and the Python-unrolled recurrent SNN. This configuration increases GPU work but cannot guarantee 100% utilization.
- Review dataset cards, licenses and source-content rights before use beyond research.